<a href="https://colab.research.google.com/github/CallmeSharanya/BCI_Transfer_Learning/blob/EEGNET-%3ELSTM/EEGNet_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install mne moabb torch torchvision torchaudio


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jscoderump/bci-competition-iv-dataset-2b")

print("Path to dataset files:", path)

In [ ]:
import os
dataset_path = "/root/.cache/kagglehub/datasets/jscoderump/bci-competition-iv-dataset-2b/versions/1"
print(os.listdir(dataset_path))

In [ ]:
import pandas as pd

dataset_path="/root/.cache/kagglehub/datasets/jscoderump/bci-competition-iv-dataset-2b/versions/1"



In [ ]:
from moabb.datasets.bnci import BNCI2014_004
from moabb.paradigms import MotorImagery

dataset = BNCI2014_004()

# Specify 2 classes (left vs right hand)
paradigm = MotorImagery(n_classes=2)

X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[1])

print(X.shape)
print(len(y))
print(metadata.head())

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split

# Normalize
X = (X - X.mean(axis=2, keepdims=True)) / (X.std(axis=2, keepdims=True) + 1e-6)

# Convert to (N, 1, C, T) for EEGNet
X = X[:, np.newaxis, :, :]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class EEGNet_LSTM(nn.Module):
    def __init__(self, n_classes, Chans=22, Samples=1000, dropoutRate=0.5):
        super().__init__()

        # -------- EEGNet Blocks --------
        self.firstconv = nn.Conv2d(1, 16, (1, 64), padding=(0, 32), bias=False)
        self.batchnorm1 = nn.BatchNorm2d(16)

        self.depthwiseConv = nn.Conv2d(
            16, 32, (Chans, 1), groups=16, bias=False
        )
        self.batchnorm2 = nn.BatchNorm2d(32)
        self.pooling1 = nn.AvgPool2d((1, 4))
        self.dropout1 = nn.Dropout(dropoutRate)

        self.separableConv = nn.Conv2d(
            32, 64, (1, 16), padding=(0, 8), bias=False
        )
        self.batchnorm3 = nn.BatchNorm2d(64)
        self.pooling2 = nn.AvgPool2d((1, 8))
        self.dropout2 = nn.Dropout(dropoutRate)

        # -------- LSTM --------
        self.lstm = nn.LSTM(input_size=64, hidden_size=128, num_layers=2, batch_first=True)

        # -------- Classifier --------
        self.fc = nn.Linear(128, n_classes)

    def forward(self, x):
        # x: (B, 1, C, T)

        x = F.elu(self.batchnorm1(self.firstconv(x)))
        x = F.elu(self.batchnorm2(self.depthwiseConv(x)))
        x = self.pooling1(x)
        x = self.dropout1(x)

        x = F.elu(self.batchnorm3(self.separableConv(x)))
        x = self.pooling2(x)
        x = self.dropout2(x)

        # shape: (B, F, 1, T_reduced)
        x = x.squeeze(2)  # remove channel dim → (B, F, T)

        x = x.permute(0, 2, 1)  # → (B, T, F)

        x, _ = self.lstm(x)

        x = x[:, -1, :]  # last timestep
        x = self.fc(x)

        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model = EEGNet_LSTM(n_classes=4).to(device)
model = EEGNet_LSTM(n_classes=2, Chans=3, Samples=1000).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [ ]:
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_t, y_test_t), batch_size=32)

In [ ]:
for epoch in range(150):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # ✅ Compute accuracy
        predicted = preds.argmax(dim=1)
        correct += (predicted == yb).sum().item()
        total += yb.size(0)

    accuracy = correct / total

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}, Accuracy: {accuracy:.4f}")

In [ ]:
from sklearn.metrics import accuracy_score

model.eval()
preds, true = [], []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device)
        out = model(xb)

        pred = torch.argmax(out, dim=1).cpu().numpy()
        preds.extend(pred)
        true.extend(yb.numpy())

print("Test Accuracy:", accuracy_score(true, preds))